In [34]:
import pandas as pd
import geopandas as gpd 
import requests 
import json
from datetime import datetime
from io import StringIO
import csv
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from shapely import wkt
from scipy.spatial.distance import cdist

import pandas as pd
import geopandas as gpd 
import requests 
import json
from datetime import datetime
from io import StringIO
import csv
import time
import os
import airqualityandclimateAPI
import statsmodels.api as sm

from statsmodels.stats.outliers_influence \
     import variance_inflation_factor as VIF
from statsmodels.stats.anova import anova_lm

from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
                         summarize,
                         poly)
from linearmodels import PanelOLS

In [15]:
from Analysis import *

In [19]:
totaldataframe = totaldata()


/Users/griffinberonio/Documents/AAE 724/AAE-724-Repo/Analysis.py:55: DtypeWarning: Columns (8,9,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  totaldf = pd.read_csv(totaldatapath)


In [20]:
# totaldata['YEAR'] = pd.to_datetime(totaldata['DATE']).dt.year
station_year_check = totaldataframe.groupby('CLIMATE_STATION_NAME')['YEAR'].nunique()
print(station_year_check)



CLIMATE_STATION_NAME
CHICAGO MIDWAY AIRPORT, IL US       10
CHICAGO PALWAUKEE AIRPORT, IL US    10
Name: YEAR, dtype: int64


# Model 1

In [21]:
xclimate = ['DailyAverageDryBulbTemperature', 'DailyAverageWindSpeed','DailyMaximumDryBulbTemperature',
       'DailyMinimumDryBulbTemperature', 'DailyPeakWindSpeed',
       'DailyPrecipitation','DailySustainedWindSpeed', 'DailySustainedWindDirection_sin', 'DailySustainedWindDirection_cos',
       'DailyAveragePrecipitation', 'DailyAveragePressureChange',
       'DailyAverageRelativeHumidity',]

y = ['arithmetic_mean']

fe = ['DATE','CLIMATE_STATION_NAME']

model_1_results = model_1(totaldataframe, xclimate, y, fe)
model_1_results

# table = pd.DataFrame({
#     "Coefficient": model_1_results.params,
#     "Std. Error": model_1_results.std_errors,
#     "t-stat": model_1_results.tstats,
#     "p-value": model_1_results.pvalues
# })

/Users/griffinberonio/Documents/AAE 724/AAE-724-Repo/Analysis.py:95: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['DATE'] = pd.to_datetime(df['DATE'])
/Users/griffinberonio/Documents/AAE 724/AAE-724-Repo/Analysis.py:99: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['YEAR'] = pd.to_numeric(df['DATE'].dt.year)


                                                index    coef  std err  \
0                                               const  2.4867   12.344   
1                      DailyAverageDryBulbTemperature  0.3159    0.160   
2                               DailyAverageWindSpeed  0.3232    0.393   
3                      DailyMaximumDryBulbTemperature  0.1442    0.029   
4                      DailyMinimumDryBulbTemperature -0.1126    0.029   
5                                  DailyPeakWindSpeed -0.1548    0.151   
6                                  DailyPrecipitation  0.0185    0.011   
7                             DailySustainedWindSpeed -0.1946    0.236   
8                     DailySustainedWindDirection_sin  0.1000    0.063   
9                     DailySustainedWindDirection_cos -1.6705    0.030   
10                          DailyAveragePrecipitation -0.1243    1.398   
11                         DailyAveragePressureChange  0.7414    0.012   
12                       DailyAverageR

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 22, but rank is 1
  warnings.warn('covariance of constraints does not have full '


,index,coef,std err,z,P>|z|
0,const,2.4867,12.344,0.201,0.840
1,DailyAverageDryBulbTemperature,0.3159,0.160,1.971,0.049
2,DailyAverageWindSpeed,0.3232,0.393,0.823,0.410
3,DailyMaximumDryBulbTemperature,0.1442,0.029,4.982,0.000
4,DailyMinimumDryBulbTemperature,-0.1126,0.029,-3.864,0.000
5,DailyPeakWindSpeed,-0.1548,0.151,-1.026,0.305
6,DailyPrecipitation,0.0185,0.011,1.643,0.100
7,DailySustainedWindSpeed,-0.1946,0.236,-0.824,0.410
8,DailySustainedWindDirection_sin,0.1000,0.063,1.582,0.114
9,DailySustainedWindDirection_cos,-1.6705,0.030,-54.981,0.000


In [22]:
model_1_table = model_1_results
model_1_table = model_1_table.rename(columns={'index':'Variable'})
model_1_table.reset_index(inplace=True)
model_1_table.head()

,index,Variable,coef,std err,z,P>|z|
0,0,const,2.4867,12.344,0.201,0.840
1,1,DailyAverageDryBulbTemperature,0.3159,0.160,1.971,0.049
2,2,DailyAverageWindSpeed,0.3232,0.393,0.823,0.410
3,3,DailyMaximumDryBulbTemperature,0.1442,0.029,4.982,0.000
4,4,DailyMinimumDryBulbTemperature,-0.1126,0.029,-3.864,0.000


In [26]:
xenergy= ['DailyAverageDryBulbTemperature', 'DailyAverageWindSpeed',
        'DailyDepartureFromNormalAverageTemperature',
        'DailyMaximumDryBulbTemperature',
       'DailyMinimumDryBulbTemperature', 'DailyPeakWindSpeed',
       'DailyPrecipitation',
       'DailySustainedWindSpeed',
       'DailyPeakWindDirection_sin', 'DailyPeakWindDirection_cos',
       'DailySustainedWindDirection_sin', 'DailySustainedWindDirection_cos',
       'DailyAverageDewPointTemperature',
       'DailyAveragePrecipitation', 'DailyAveragePressureChange',
       'DailyAverageRelativeHumidity', 'DailyAverageSeaLevelPressure',
       'DailyAverageStationPressure', 'DailyAverageWetBulbTemperature',
       'DailyAverageWindGustSpeed', 'DailyAverageWindDirection_sin',
       'DailyAverageWindDirection_cos']

xenergyandrenewables = xenergy + ['Number', 'Capacity']

yenergy = ['Demand']
yfossil = ['Gross Load (MWh)']
yaqi = ['aqi']

fe = ['CLIMATE_STATION_NAME','YEAR']
feenergy = ['CLIMATE_STATION_NAME','MONTH']



In [28]:
model_1_panel_results = model_1_panel(totaldataframe, xenergy, y, fe)


/Users/griffinberonio/Documents/AAE 724/AAE-724-Repo/Analysis.py:149: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors='coerce')
/opt/anaconda3/lib/python3.13/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


                          PanelOLS Estimation Summary                           
Dep. Variable:        arithmetic_mean   R-squared:                        0.1163
Estimator:                   PanelOLS   R-squared (Between):             -2520.4
No. Observations:              186761   R-squared (Within):               0.1150
Date:                Wed, Mar 11 2026   R-squared (Overall):             -1740.6
Time:                        22:00:44   Log-likelihood                -5.924e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      1116.8
Entities:                           2   P-value                           0.0000
Avg Obs:                    9.338e+04   Distribution:               F(22,186728)
Min Obs:                    9.025e+04                                           
Max Obs:                    9.651e+04   F-statistic (robust):         -9.772e+15
                            

In [40]:
from great_tables import GT

from linearmodels.panel.results import PanelEffectsResults

def make_table(results, titles, filename):

    # Detect PanelOLS results object
    if isinstance(results, PanelEffectsResults):

        table = pd.DataFrame({
            "Coefficient": results.params,
            "Std. Error": results.std_errors,
            "t-stat": results.tstats,
            "p-value": results.pvalues
        }).round(4)

        table = table.reset_index()
        table = table.rename(columns={"index": "Variable"})

    # If a dataframe is passed directly
    elif isinstance(results, pd.DataFrame):

        table = results.copy()

        if "Variable" not in table.columns:
            table = table.reset_index()
            table = table.rename(columns={"index": "Variable"})

    else:
        raise ValueError("Unsupported results type")

    # Create formatted table
    gt_table = (
        GT(table)
        .tab_header(
            title="Panel Regression Results",
            subtitle=titles
        )
        .opt_table_font(font="Times New Roman")
    )

    print("Saving table...")

    tablepath = "/Users/griffinberonio/Documents/AAE 724/"

    gt_table.save(f"{tablepath}{filename}.png")

    print("Table saved successfully")


In [39]:
make_table(model_1_panel_results,'AVG PM 2.5 on Climate Variables','model_1_panel.png')
# model_1_panel_results.params

Saving table...
Table saved successfully


In [45]:
model_2_results = model_1_panel(totaldataframe, xenergy, yaqi,fe)

/Users/griffinberonio/Documents/AAE 724/AAE-724-Repo/Analysis.py:149: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors='coerce')
/opt/anaconda3/lib/python3.13/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


                          PanelOLS Estimation Summary                           
Dep. Variable:                    aqi   R-squared:                        0.2170
Estimator:                   PanelOLS   R-squared (Between):             -485.59
No. Observations:              172037   R-squared (Within):               0.2116
Date:                Wed, Mar 11 2026   R-squared (Overall):             -421.18
Time:                        22:13:35   Log-likelihood                -7.092e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      2166.8
Entities:                           2   P-value                           0.0000
Avg Obs:                    8.602e+04   Distribution:               F(22,172004)
Min Obs:                    8.018e+04                                           
Max Obs:                    9.186e+04   F-statistic (robust):           1.08e+15
                            

In [47]:
make_table(model_2_results, 'AQI on Climate Variables', 'model_2_aqi.png')

Saving table...
Table saved successfully


In [49]:
model_3_results = model_3_energy_climate(totaldataframe, xenergy, yenergy, feenergy)


                          PanelOLS Estimation Summary                           
Dep. Variable:                 Demand   R-squared:                        0.0558
Estimator:                   PanelOLS   R-squared (Between):             -4.9780
No. Observations:              147325   R-squared (Within):               0.0661
Date:                Wed, Mar 11 2026   R-squared (Overall):             -4.8942
Time:                        22:14:34   Log-likelihood                -1.995e+06
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      395.66
Entities:                           2   P-value                           0.0000
Avg Obs:                    7.366e+04   Distribution:               F(22,147290)
Min Obs:                    7.169e+04                                           
Max Obs:                    7.564e+04   F-statistic (robust):         -1.706e+16
                            

In [50]:
make_table(model_3_results, 'Daily Energy Demand on Climate Variables','model_3_energy.png')

Saving table...
Table saved successfully


In [51]:
model_4_results = model_3_energy_climate(totaldataframe,xenergy, yfossil, feenergy)


                          PanelOLS Estimation Summary                           
Dep. Variable:       Gross Load (MWh)   R-squared:                        0.0684
Estimator:                   PanelOLS   R-squared (Between):             -57.501
No. Observations:              187615   R-squared (Within):               0.1562
Date:                Wed, Mar 11 2026   R-squared (Overall):             -51.597
Time:                        22:21:30   Log-likelihood                -2.284e+06
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      626.23
Entities:                           2   P-value                           0.0000
Avg Obs:                    9.381e+04   Distribution:               F(22,187580)
Min Obs:                    9.048e+04                                           
Max Obs:                    9.713e+04   F-statistic (robust):         -9.337e+15
                            

In [52]:
make_table(model_4_results,'Fossil Fuel Generation on Climate Data','model_4_fossil.png')

Saving table...
Table saved successfully


In [54]:
model_5_results = model_3_energy_climate(totaldataframe, xenergyandrenewables,y,feenergy)
make_table(model_5_results, 'AVG PM2.5 on Renewables and Climate', 'model_5_renewables_climate.png')

                          PanelOLS Estimation Summary                           
Dep. Variable:        arithmetic_mean   R-squared:                        0.1245
Estimator:                   PanelOLS   R-squared (Between):             -2769.3
No. Observations:              186761   R-squared (Within):               0.1070
Date:                Wed, Mar 11 2026   R-squared (Overall):             -1912.5
Time:                        22:27:09   Log-likelihood                -5.898e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      1106.3
Entities:                           2   P-value                           0.0000
Avg Obs:                    9.338e+04   Distribution:               F(24,186724)
Min Obs:                    9.025e+04                                           
Max Obs:                    9.651e+04   F-statistic (robust):         -8.493e+15
                            

In [55]:
model_6_results = model_3_energy_climate(totaldataframe,xenergyandrenewables,yaqi,fe)
make_table(model_6_results, 'AQI on Renewables and Climate Variables', 'model_6_aqi_renewables.png')


                          PanelOLS Estimation Summary                           
Dep. Variable:                    aqi   R-squared:                        0.2170
Estimator:                   PanelOLS   R-squared (Between):             -486.04
No. Observations:              172037   R-squared (Within):               0.2115
Date:                Wed, Mar 11 2026   R-squared (Overall):             -421.58
Time:                        22:35:08   Log-likelihood                -7.092e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      1986.4
Entities:                           2   P-value                           0.0000
Avg Obs:                    8.602e+04   Distribution:               F(24,172002)
Min Obs:                    8.018e+04                                           
Max Obs:                    9.186e+04   F-statistic (robust):          3.054e+16
                            